# Bước 0: Làm sạch và Chuẩn hóa Dữ liệu (Data Cleaning)
Đây là bước tiên quyết trong bất kỳ dự án Machine Learning thực tế nào. Quá trình này giúp:
1. **Phát hiện và loại bỏ ảnh lỗi (Corrupted Images):** Tránh việc mô hình bị crash trong lúc huấn luyện do không đọc được file.
2. **Chuẩn hóa định dạng:** Chuyển tất cả ảnh về đuôi `.jpg`.
3. **Đổi tên đồng loạt (Renaming):** Dọn dẹp các file có tên lộn xộn (ví dụ: `download(1).png`, `IMG_123.jpg`) thành định dạng quy chuẩn (ví dụ: `plastic_0001.jpg`).

In [2]:
import os
import cv2
from tqdm.notebook import tqdm

def clean_and_rename_dataset(base_dir):
    if not os.path.exists(base_dir):
        print(f"Thư mục {base_dir} không tồn tại.")
        return
        
    classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    classes.sort()
    
    total_corrupted = 0
    total_processed = 0
    
    for class_name in classes:
        class_path = os.path.join(base_dir, class_name)
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"\nĐang xử lý thư mục: {class_name} (Có {len(files)} file)")
        
        # BƯỚC 1: Lọc ảnh lỗi và Đổi sang tên tạm thời (Tránh trùng lặp tên cũ)
        valid_files = []
        for idx, f in enumerate(files):
            file_path = os.path.join(class_path, f)
            
            # Kiểm tra xem ảnh có đọc được không
            try:
                img = cv2.imread(file_path)
                if img is None:
                    raise ValueError("Cannot decode image")
                
                # Nếu đọc thành công, đổi tên thành tên TẠM THỜI
                temp_name = f"temp_{class_name}_{idx}.jpg"
                temp_path = os.path.join(class_path, temp_name)
                os.rename(file_path, temp_path)
                valid_files.append(temp_path)
                
            except Exception as e:
                # Ảnh lỗi -> Xóa luôn
                print(f"  [!] Phát hiện ảnh lỗi, đang xóa: {f}")
                os.remove(file_path)
                total_corrupted += 1
        
        # BƯỚC 2: Đổi tên từ tạm thời sang ĐỊNH DẠNG CHUẨN
        # Ví dụ: plastic_0001.jpg
        for final_idx, temp_path in enumerate(valid_files, 1):
            final_name = f"{class_name}_{final_idx:04d}.jpg"
            final_path = os.path.join(class_path, final_name)
            os.rename(temp_path, final_path)
            total_processed += 1
            
    print("\n" + "="*50)
    print("✅ HOÀN TẤT DỌN DẸP DỮ LIỆU!")
    print(f"- Đã xóa {total_corrupted} ảnh bị lỗi/hỏng.")
    print(f"- Đã chuẩn hóa và đổi tên {total_processed} ảnh hợp lệ.")
    print("="*50)

### Bắt đầu chạy Dọn dẹp cho tập Train và Validation

In [4]:
print("--- XỬ LÝ TẬP HUẤN LUYỆN (TRAIN) ---")
clean_and_rename_dataset("../dataset/train")

print("\n--- XỬ LÝ TẬP ĐÁNH GIÁ (VALIDATION) ---")
clean_and_rename_dataset("../dataset/validation")

--- XỬ LÝ TẬP HUẤN LUYỆN (TRAIN) ---

Đang xử lý thư mục: cardboard (Có 3500 file)

Đang xử lý thư mục: glass (Có 3500 file)

Đang xử lý thư mục: metal (Có 3500 file)

Đang xử lý thư mục: organic (Có 3500 file)

Đang xử lý thư mục: paper (Có 3500 file)

Đang xử lý thư mục: plastic (Có 3500 file)

Đang xử lý thư mục: trash (Có 2702 file)

✅ HOÀN TẤT DỌN DẸP DỮ LIỆU!
- Đã xóa 0 ảnh bị lỗi/hỏng.
- Đã chuẩn hóa và đổi tên 23702 ảnh hợp lệ.

--- XỬ LÝ TẬP ĐÁNH GIÁ (VALIDATION) ---

Đang xử lý thư mục: cardboard (Có 46 file)

Đang xử lý thư mục: glass (Có 41 file)

Đang xử lý thư mục: metal (Có 53 file)

Đang xử lý thư mục: organic (Có 54 file)

Đang xử lý thư mục: paper (Có 84 file)

Đang xử lý thư mục: plastic (Có 65 file)

Đang xử lý thư mục: trash (Có 21 file)

✅ HOÀN TẤT DỌN DẸP DỮ LIỆU!
- Đã xóa 0 ảnh bị lỗi/hỏng.
- Đã chuẩn hóa và đổi tên 364 ảnh hợp lệ.
